<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/11_NeuroFHIR_QC_Reviewer_Application_UI_COLAB_BUILD_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/11_NeuroFHIR_QC_Reviewer_Application_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Notebook 11
## Reviewer Application UI

This notebook builds the next competition stage: a reviewer-facing application over the completed NeuroFHIR-QC evidence.

### Implemented screens

1. Launch and Study Context
2. Patient Timeline
3. MRI Review
4. Longitudinal Analysis
5. Human Review
6. FHIR Audit

It also generates a read-only FastAPI evidence service, Docker files, a GitHub Pages workflow, a production React build, an audit, and a source ZIP.

### Evidence boundary

The application uses public de-identified research imaging and synthetic FHIR R4 context. It does not claim production SMART OAuth, hospital deployment, clinical validation, real clinician review, or human usability results.

### Colab build correction in this version

`npm install` and the Vite build now run under `/content` rather than
Google Drive. This avoids the `esbuild EACCES` error caused by native
executables stored on the mounted Drive filesystem. The generated
`package-lock.json` and successful `dist/` build are copied back to the
project afterward.


In [1]:
# Cell 1 — Mount Drive and enforce the Notebook 10 gate

from __future__ import annotations

import base64
import hashlib
import json
import math
import re
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
NOTEBOOK_FILENAME = "11_NeuroFHIR_QC_Reviewer_Application_UI.ipynb"
NOTEBOOK_SAVE_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_FILENAME
PROJECT_CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"
NB10_AUDIT_PATH = PROJECT_ROOT / "evaluation/results/notebook_10_competition_artifacts_audit.json"
NB09_ROOT = PROJECT_ROOT / "evaluation/results/notebook_09_evaluation"
NB09_SEGMENTATION_PATH = NB09_ROOT / "segmentation_evaluation.json"
NB09_ROBUSTNESS_PATH = NB09_ROOT / "robustness_qc_evaluation.json"
NB09_LONGITUDINAL_PATH = NB09_ROOT / "longitudinal_evaluation.json"
NB09_INTEROPERABILITY_PATH = NB09_ROOT / "fhir_interoperability_evaluation.json"
NB09_WORKFLOW_PATH = NB09_ROOT / "workflow_safety_evaluation.json"
NB09_SCORECARD_PATH = NB09_ROOT / "competition_evaluation_scorecard.json"
NB07_ROOT = PROJECT_ROOT / "submission/fhir_resources/notebook_07"
NB07_MANIFEST_PATH = NB07_ROOT / "fhir_evidence_manifest.json"
NB07_RESOURCE_ROOT = NB07_ROOT / "resources"
NB08_ROOT = PROJECT_ROOT / "submission/fhir_resources/notebook_08"
NB08_TRANSITIONS_PATH = PROJECT_ROOT / "evaluation/results/notebook_08_human_review/review_transition_report.json"
NB08_REVIEW_RESOURCE_ROOT = NB08_ROOT / "reviewed_resources"
SOURCE_INDEX_PATH = PROJECT_ROOT / "data/synthetic_fhir/notebook_01/resource_index.json"
APP_ROOT = PROJECT_ROOT / "app"
FRONTEND_ROOT = APP_ROOT / "frontend"
BACKEND_ROOT = APP_ROOT / "backend"
PUBLIC_ROOT = FRONTEND_ROOT / "public"
DATA_ROOT = PUBLIC_ROOT / "data"
ASSET_ROOT = PUBLIC_ROOT / "assets"
BACKEND_DATA_ROOT = BACKEND_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "evaluation/results/notebook_11_reviewer_application_ui"
SUBMISSION_BUILD_ROOT = PROJECT_ROOT / "submission/reviewer_application"
DOC_ROOT = PROJECT_ROOT / "docs"
APP_DATA_PATH = DATA_ROOT / "app_data.json"
SCREEN_MATRIX_PATH = OUTPUT_ROOT / "screen_implementation_matrix.json"
BUILD_REPORT_PATH = OUTPUT_ROOT / "frontend_build_report.json"
API_REPORT_PATH = OUTPUT_ROOT / "backend_api_report.json"
FILE_INVENTORY_PATH = OUTPUT_ROOT / "application_file_inventory.json"
AUDIT_JSON_PATH = PROJECT_ROOT / "evaluation/results/notebook_11_reviewer_application_ui_audit.json"
AUDIT_MD_PATH = DOC_ROOT / "NOTEBOOK_11_REVIEWER_APPLICATION_UI.md"
DEPLOYMENT_GUIDE_PATH = DOC_ROOT / "REVIEWER_APPLICATION_DEPLOYMENT.md"
ZIP_PATH = PROJECT_ROOT / "submission/NeuroFHIR_QC_Reviewer_Application_Source.zip"

for folder in (OUTPUT_ROOT, SUBMISSION_BUILD_ROOT, DOC_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write("\n")
    temporary.replace(path)

def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

required_paths = [
    PROJECT_CONFIG_PATH, NOTEBOOK_MANIFEST_PATH, NB10_AUDIT_PATH,
    NB09_SEGMENTATION_PATH, NB09_ROBUSTNESS_PATH, NB09_LONGITUDINAL_PATH,
    NB09_INTEROPERABILITY_PATH, NB09_WORKFLOW_PATH, NB09_SCORECARD_PATH,
    NB07_MANIFEST_PATH, NB08_TRANSITIONS_PATH, SOURCE_INDEX_PATH,
]
missing = [str(path) for path in required_paths if not path.exists() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError("Notebook 11 prerequisites are missing:\n" + "\n".join(f" - {path}" for path in missing))

project_config = load_json(PROJECT_CONFIG_PATH)
notebook_manifest = load_json(NOTEBOOK_MANIFEST_PATH)
nb10_audit = load_json(NB10_AUDIT_PATH)
if nb10_audit.get("status") != "completed":
    raise RuntimeError("Notebook 10 audit is not completed.")

for key in (
    "fhir_validation_pass_rate",
    "fhir_transaction_success_rate",
    "fhir_transaction_entry_success_rate",
    "critical_field_preservation_rate",
    "provenance_completeness_rate",
):
    if nb10_audit.get("technical_metrics", {}).get(key) != 1.0:
        raise AssertionError(f"Notebook 10 gate failed: {key}")

CASE_ORDER = ("stable", "progression", "low-confidence")
print("=" * 104)
print("✅ Notebook 10 completion gate passed")
print("✅ Reviewer application generation may begin")
print("⚠️ Synthetic FHIR and public research imaging only")
print("=" * 104)

Mounted at /content/drive
✅ Notebook 10 completion gate passed
✅ Reviewer application generation may begin
⚠️ Synthetic FHIR and public research imaging only


In [2]:
# Cell 2 — Extract the verified reviewer-application source template

TEMPLATE_BASE64 = "..."
TEMPLATE_ZIP_PATH = Path("/tmp/neurofhir_ui_template.zip")
TEMPLATE_EXTRACT_ROOT = Path("/tmp/neurofhir_ui_template")

TEMPLATE_ZIP_PATH.write_bytes(base64.b64decode(TEMPLATE_BASE64))
if TEMPLATE_EXTRACT_ROOT.exists():
    shutil.rmtree(TEMPLATE_EXTRACT_ROOT)
TEMPLATE_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(TEMPLATE_ZIP_PATH, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise AssertionError(f"Embedded template ZIP failed integrity: {bad_member}")
    archive.extractall(TEMPLATE_EXTRACT_ROOT)

if APP_ROOT.exists():
    shutil.rmtree(APP_ROOT)
shutil.copytree(TEMPLATE_EXTRACT_ROOT / "app", APP_ROOT)

workflow_source = TEMPLATE_EXTRACT_ROOT / ".github/workflows/deploy-reviewer-ui.yml"
workflow_destination = PROJECT_ROOT / ".github/workflows/deploy-reviewer-ui.yml"
workflow_destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(workflow_source, workflow_destination)

for folder in (DATA_ROOT, ASSET_ROOT, BACKEND_DATA_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 104)
print("✅ React, FastAPI, Docker, and GitHub Pages source extracted")
print(f"✅ Application root: {APP_ROOT}")
print("=" * 104)

✅ React, FastAPI, Docker, and GitHub Pages source extracted
✅ Application root: /content/drive/MyDrive/neurofhir-qc/app


In [3]:
# Cell 3 — Build reviewer application data from executed artifacts

segmentation = load_json(NB09_SEGMENTATION_PATH)
robustness = load_json(NB09_ROBUSTNESS_PATH)
longitudinal = load_json(NB09_LONGITUDINAL_PATH)
interoperability = load_json(NB09_INTEROPERABILITY_PATH)
workflow = load_json(NB09_WORKFLOW_PATH)
scorecard = load_json(NB09_SCORECARD_PATH)
fhir_manifest = load_json(NB07_MANIFEST_PATH)
transition_report = load_json(NB08_TRANSITIONS_PATH)
source_index = load_json(SOURCE_INDEX_PATH)

def ref(resource: dict[str, Any]) -> str:
    return f"{resource['resourceType']}/{resource['id']}"

def case_resources(case_id: str) -> list[dict[str, Any]]:
    resources = []
    for row in source_index.get("resources", []):
        if row.get("case_id") != case_id:
            continue
        path = PROJECT_ROOT / row["relative_path"]
        if not path.exists():
            raise FileNotFoundError(path)
        resources.append(load_json(path))
    return resources

def first(resources: list[dict[str, Any]], resource_type: str) -> dict[str, Any] | None:
    return next((resource for resource in resources if resource.get("resourceType") == resource_type), None)

def all_of(resources: list[dict[str, Any]], resource_type: str) -> list[dict[str, Any]]:
    return [resource for resource in resources if resource.get("resourceType") == resource_type]

seg_rows = {row["case_id"]: row for row in segmentation.get("rows", [])}
qc_rows = {row["case_id"]: row for row in robustness.get("case_qc_rows", [])}
long_rows = {row["case_id"]: row for row in longitudinal.get("rows", [])}
fhir_rows = {row["case_id"]: row for row in fhir_manifest.get("cases", [])}
events_by_case = {case_id: [] for case_id in CASE_ORDER}
for event in transition_report.get("events", []):
    events_by_case[event["case_id"]].append(event)

def copy_visuals(case_id: str) -> list[str]:
    roots = [
        PROJECT_ROOT / "evaluation/results/notebook_04_segmentation_and_volumetry",
        PROJECT_ROOT / "evaluation/results/notebook_05_trust_and_robustness",
        PROJECT_ROOT / "evaluation/results/notebook_09_evaluation/plots",
    ]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob("*.png"):
            lower = path.name.lower()
            score = 0
            if case_id in lower:
                score += 5
            if any(token in lower for token in ("overlay", "mask", "segmentation", "montage")):
                score += 4
            if any(token in lower for token in ("qc", "volume", "longitudinal")):
                score += 2
            if score:
                candidates.append((score, path))
    selected = []
    seen = set()
    for _, source in sorted(candidates, key=lambda item: (-item[0], str(item[1]))):
        digest = sha256_file(source)
        if digest in seen:
            continue
        destination = ASSET_ROOT / f"{case_id}_{len(selected)+1}_{source.name}"
        shutil.copy2(source, destination)
        selected.append(f"./assets/{destination.name}")
        seen.add(digest)
        if len(selected) == 3:
            break
    return selected

cases = []
for case_id in CASE_ORDER:
    resources = case_resources(case_id)
    patient = first(resources, "Patient")
    condition = first(resources, "Condition")
    studies = sorted(all_of(resources, "ImagingStudy"), key=lambda row: str(row.get("started") or row.get("id")))
    prior = first(resources, "Observation")
    if patient is None or condition is None or len(studies) != 2:
        raise AssertionError(f"{case_id}: incomplete source context")

    seg = seg_rows[case_id]
    qc = qc_rows[case_id]
    long_row = long_rows[case_id]
    fhir_row = fhir_rows[case_id]
    review_events = events_by_case[case_id]
    final_event = next((event for event in reversed(review_events) if bool(event.get("final_event_for_case"))), None)
    patient_name = patient.get("name", [{}])[0].get("text") or "Synthetic Patient"
    condition_name = condition.get("code", {}).get("text") or condition.get("code", {}).get("coding", [{}])[0].get("display") or "Synthetic neurological research condition"

    cases.append({
        "case_id": case_id,
        "display_name": case_id.replace("-", " ").title(),
        "patient": {
            "reference": ref(patient),
            "display": patient_name,
            "synthetic": True,
            "gender": patient.get("gender"),
            "birth_date": patient.get("birthDate"),
        },
        "condition": {"reference": ref(condition), "display": condition_name},
        "imaging_studies": [{
            "reference": ref(study),
            "description": study.get("description") or study.get("procedureCode", [{}])[0].get("text") or "MRI study",
            "started": study.get("started"),
            "status": study.get("status"),
        } for study in studies],
        "prior_observation": {
            "reference": ref(prior) if prior else None,
            "status": prior.get("status") if prior else "final",
        },
        "segmentation": {
            "dice": seg.get("dice"),
            "sensitivity": seg.get("sensitivity"),
            "precision": seg.get("precision"),
            "hd95_mm": seg.get("hd95_mm"),
            "predicted_volume_ml": seg.get("predicted_volume_ml"),
            "reference_volume_ml": seg.get("reference_volume_ml"),
            "absolute_volume_error_ml": seg.get("absolute_volume_error_ml"),
            "inference_seconds": seg.get("inference_seconds"),
        },
        "qc": {
            "score": qc.get("qc_score"),
            "category": qc.get("qc_category"),
            "min_mask_dice": qc.get("effective_min_mask_dice"),
            "max_relative_volume_change": qc.get("effective_max_relative_volume_change"),
            "max_boundary_hd95_mm": qc.get("effective_max_boundary_hd95_mm"),
            "plausibility_score": qc.get("plausibility_score"),
            "provenance_completeness_score": qc.get("provenance_completeness_score"),
        },
        "longitudinal": {
            "prior_volume_ml": long_row.get("prior_reviewed_baseline_volume_ml"),
            "current_volume_ml": long_row.get("selected_current_ai_volume_ml"),
            "absolute_change_ml": long_row.get("absolute_change_ml"),
            "percent_change": long_row.get("percent_change"),
            "display_label": long_row.get("longitudinal_display_label"),
            "interpretation": long_row.get("longitudinal_interpretation"),
            "interpretation_withheld": bool(long_row.get("interpretation_withheld")),
            "scenario_alignment_passed": bool(long_row.get("scenario_alignment_passed")),
        },
        "review": {
            "events": review_events,
            "final_decision": final_event.get("decision") if final_event else None,
            "final_observation_status": final_event.get("after_observation_status") if final_event else "preliminary",
            "final_report_status": final_event.get("after_report_status") if final_event else "preliminary",
            "final_task_status": final_event.get("after_task_status") if final_event else "requested",
            "correction_required_preserved": any(event.get("decision") == "correction-required" for event in review_events),
        },
        "fhir": {
            "source_context_resources": fhir_row.get("source_context_resources", []),
            "generated_resources": fhir_row.get("resources", {}),
            "transaction_bundle_file": fhir_row.get("transaction_bundle_file"),
        },
        "images": copy_visuals(case_id),
    })

representative_resources = {}
for case_row in fhir_manifest.get("cases", []):
    for resource_type, reference in case_row.get("resources", {}).items():
        resource_id = reference.split("/", 1)[1]
        path = NB07_RESOURCE_ROOT / f"{resource_type}-{resource_id}.json"
        if path.exists():
            representative_resources[reference] = load_json(path)
if NB08_REVIEW_RESOURCE_ROOT.exists():
    for path in sorted(NB08_REVIEW_RESOURCE_ROOT.glob("*.json")):
        resource = load_json(path)
        representative_resources[ref(resource)] = resource

app_data = {
    "generated_utc": utc_now(),
    "project": {
        "name": "NeuroFHIR-QC",
        "title": "Trustworthy Longitudinal Neuroimaging AI in FHIR",
        "tagline": "Quality-scored, provenance-aware, human-reviewed longitudinal neuroimaging evidence.",
        "environment": "Research demonstration",
        "data_boundary": "Public de-identified research MRI linked only to synthetic FHIR R4 patient context.",
        "repository_url": "https://github.com/SANGHATI23/neurofhir-qc",
        "fhir_server": interoperability.get("server_base_url", "https://hapi.fhir.org/baseR4"),
        "fhir_version": interoperability.get("fhir_version", "4.0.1"),
        "smart_status": "SMART-compatible reviewer path; production OAuth launch is not implemented or claimed.",
    },
    "summary_metrics": {
        "mean_dice": segmentation.get("mean_whole_tumor_dice"),
        "mean_volume_error_ml": segmentation.get("mean_absolute_volume_error_ml"),
        "mean_inference_seconds": segmentation.get("mean_inference_seconds"),
        "robustness_runs": robustness.get("standard_perturbation_run_count", 0) + robustness.get("challenge_run_count", 0),
        "scenario_alignment": f"{longitudinal.get('scenario_alignment_pass_count', 0)}/{longitudinal.get('case_count', 0)}",
        "fhir_validation": f"{interoperability.get('validation_pass_count', 0)}/{interoperability.get('validation_target_count', 0)}",
        "transactions": f"{interoperability.get('transaction_success_count', 0)}/{interoperability.get('transaction_count', 0)}",
        "transaction_entries": f"{interoperability.get('successful_entry_count', 0)}/{interoperability.get('submitted_entry_count', 0)}",
        "readback_preservation": interoperability.get("critical_field_preservation_rate"),
        "review_events": workflow.get("review_transition_event_count"),
    },
    "cases": cases,
    "representative_fhir_resources": representative_resources,
    "evaluation_domains": scorecard.get("rows", []),
    "limitations": [
        "Three-case executable demonstration benchmark; not independent external validation.",
        "Public images are linked to synthetic FHIR context, not real longitudinal clinical records.",
        "QC scores are engineering workflow signals, not clinical safety probabilities.",
        "The reviewer is synthetic; clinician agreement and usability are not claimed.",
        "The public HAPI R4 sandbox is not a production hospital environment.",
        "Production SMART OAuth and clinical deployment are not claimed.",
    ],
}
write_json(APP_DATA_PATH, app_data)
BACKEND_DATA_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(APP_DATA_PATH, BACKEND_DATA_ROOT / "app_data.json")

if len(cases) != 3:
    raise AssertionError("Application data must contain three cases.")
if len(representative_resources) < 13:
    raise AssertionError("Too few representative FHIR resources.")

print("=" * 104)
print("✅ Application data generated")
print(f"✅ Cases: {len(cases)}")
print(f"✅ Representative FHIR resources: {len(representative_resources)}")
for item in cases:
    print(f" - {item['case_id']}: QC={item['qc']['category']}, review={item['review']['final_decision']}, visuals={len(item['images'])}")
print("=" * 104)

✅ Application data generated
✅ Cases: 3
✅ Representative FHIR resources: 17
 - stable: QC=High confidence, review=accepted, visuals=3
 - progression: QC=High confidence, review=accepted, visuals=3
 - low-confidence: QC=Manual review required, review=rejected, visuals=3


In [4]:
# Cell 4 — Generate deployment documentation and validate source

DEPLOYMENT_GUIDE_PATH.write_text(
    "# NeuroFHIR-QC Reviewer Application Deployment\n\n"
    "## GitHub Pages\n\n"
    "1. Commit `app/frontend/`.\n"
    "2. Commit `.github/workflows/deploy-reviewer-ui.yml`.\n"
    "3. Open GitHub **Settings → Pages**.\n"
    "4. Select **GitHub Actions** as the source.\n"
    "5. Push to `main` or run the workflow manually.\n"
    "6. Record the public URL in the AMIA package.\n\n"
    "## Competition capture\n\n"
    "Capture all six screens, especially low-confidence warning, interpretation withheld, correction-required, rejection, FHIR JSON, validation, transaction, and read-back evidence.\n\n"
    "## Privacy\n\n"
    "Do not publish account details, email, tokens, credentials, or real patient identifiers.\n",
    encoding="utf-8",
)

screen_rows = [
    {"screen": "Launch and Study Context", "implemented": True},
    {"screen": "Patient Timeline", "implemented": True},
    {"screen": "MRI Review", "implemented": True},
    {"screen": "Longitudinal Analysis", "implemented": True},
    {"screen": "Human Review", "implemented": True},
    {"screen": "FHIR Audit", "implemented": True},
]
write_json(SCREEN_MATRIX_PATH, {
    "generated_utc": utc_now(),
    "screen_count": len(screen_rows),
    "implementation_rate": 1.0,
    "screens": screen_rows,
})

backend_python_files = sorted(BACKEND_ROOT.rglob("*.py"))
for path in backend_python_files:
    compile(path.read_text(encoding="utf-8"), str(path), "exec")

required_source = [
    FRONTEND_ROOT / "package.json",
    FRONTEND_ROOT / "src/App.tsx",
    FRONTEND_ROOT / "src/styles.css",
    BACKEND_ROOT / "app/main.py",
    PROJECT_ROOT / ".github/workflows/deploy-reviewer-ui.yml",
    APP_DATA_PATH,
]
missing_source = [str(path) for path in required_source if not path.exists() or path.stat().st_size == 0]
if missing_source:
    raise AssertionError("Application source missing:\n" + "\n".join(f" - {path}" for path in missing_source))

write_json(API_REPORT_PATH, {
    "generated_utc": utc_now(),
    "backend_python_file_count": len(backend_python_files),
    "backend_syntax_passed": True,
    "writeback_endpoint_exposed": False,
    "endpoints": [
        "GET /api/health",
        "GET /api/app-data",
        "GET /api/cases",
        "GET /api/cases/{case_id}",
        "GET /api/fhir/{resource_type}/{resource_id}",
        "GET /api/fhir-live/{resource_type}/{resource_id}",
        "POST /api/review-preview",
    ],
})

print("=" * 104)
print("✅ Six-screen matrix generated")
print("✅ FastAPI source syntax passed")
print("✅ Deployment guide generated")
print("=" * 104)

✅ Six-screen matrix generated
✅ FastAPI source syntax passed
✅ Deployment guide generated


In [5]:
# Cell 5 — Install and build on Colab's local filesystem, then copy results to Drive

# Google Drive is mounted with filesystem behavior that may block execution
# of native npm binaries such as esbuild. Therefore, dependency installation
# and the Vite production build are performed under /content, where executable
# permissions work normally. Only source, package-lock.json, and build outputs
# are copied back to the project in Drive.

node_result = subprocess.run(
    ["node", "--version"],
    capture_output=True,
    text=True,
)
npm_result = subprocess.run(
    ["npm", "--version"],
    capture_output=True,
    text=True,
)
if node_result.returncode != 0 or npm_result.returncode != 0:
    raise RuntimeError(
        "Node.js/npm are not available in this Colab runtime."
    )

node_version = node_result.stdout.strip()
npm_version = npm_result.stdout.strip()

LOCAL_BUILD_ROOT = Path(
    "/content/neurofhir_qc_reviewer_frontend_build"
)
LOCAL_FRONTEND_ROOT = LOCAL_BUILD_ROOT / "frontend"

if LOCAL_BUILD_ROOT.exists():
    shutil.rmtree(LOCAL_BUILD_ROOT)
LOCAL_FRONTEND_ROOT.mkdir(parents=True, exist_ok=True)

# Copy only application source/configuration. Never copy Drive node_modules
# or a previous dist directory into the local build workspace.
ignored_names = {
    "node_modules",
    "dist",
    ".vite",
    ".npm",
}
for source_path in FRONTEND_ROOT.rglob("*"):
    relative_path = source_path.relative_to(FRONTEND_ROOT)

    if any(part in ignored_names for part in relative_path.parts):
        continue

    destination_path = LOCAL_FRONTEND_ROOT / relative_path

    if source_path.is_dir():
        destination_path.mkdir(parents=True, exist_ok=True)
    elif source_path.is_file():
        destination_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, destination_path)

required_local_sources = [
    LOCAL_FRONTEND_ROOT / "package.json",
    LOCAL_FRONTEND_ROOT / "vite.config.ts",
    LOCAL_FRONTEND_ROOT / "index.html",
    LOCAL_FRONTEND_ROOT / "src/App.tsx",
    LOCAL_FRONTEND_ROOT / "src/styles.css",
    LOCAL_FRONTEND_ROOT / "public/data/app_data.json",
]
missing_local_sources = [
    str(path)
    for path in required_local_sources
    if not path.exists() or path.stat().st_size == 0
]
if missing_local_sources:
    raise FileNotFoundError(
        "Local frontend build workspace is incomplete:\n"
        + "\n".join(
            f" - {path}"
            for path in missing_local_sources
        )
    )

install_started = time.perf_counter()
install_process = subprocess.run(
    [
        "npm",
        "install",
        "--no-audit",
        "--no-fund",
    ],
    cwd=LOCAL_FRONTEND_ROOT,
    capture_output=True,
    text=True,
    timeout=600,
)
install_seconds = time.perf_counter() - install_started

if install_process.returncode != 0:
    raise RuntimeError(
        "npm install failed in Colab local storage:\n"
        + install_process.stdout[-2500:]
        + "\n"
        + install_process.stderr[-5000:]
    )

local_esbuild_candidates = [
    LOCAL_FRONTEND_ROOT
    / "node_modules/@esbuild/linux-x64/bin/esbuild",
    LOCAL_FRONTEND_ROOT
    / "node_modules/esbuild/bin/esbuild",
]
for executable in local_esbuild_candidates:
    if executable.exists():
        executable.chmod(
            executable.stat().st_mode | 0o111
        )

build_started = time.perf_counter()
build_process = subprocess.run(
    ["npm", "run", "build"],
    cwd=LOCAL_FRONTEND_ROOT,
    capture_output=True,
    text=True,
    timeout=600,
)
build_seconds = time.perf_counter() - build_started

if build_process.returncode != 0:
    raise RuntimeError(
        "Frontend build failed in Colab local storage:\n"
        + build_process.stdout[-4000:]
        + "\n"
        + build_process.stderr[-6000:]
    )

local_dist_root = LOCAL_FRONTEND_ROOT / "dist"
required_local_dist = [
    local_dist_root / "index.html",
    local_dist_root / "data/app_data.json",
]
missing_local_dist = [
    str(path)
    for path in required_local_dist
    if not path.exists() or path.stat().st_size == 0
]
if missing_local_dist:
    raise AssertionError(
        "Local production build is incomplete:\n"
        + "\n".join(
            f" - {path}"
            for path in missing_local_dist
        )
    )

# Preserve package-lock.json for deterministic GitHub Actions npm ci builds.
local_package_lock = LOCAL_FRONTEND_ROOT / "package-lock.json"
if not local_package_lock.exists():
    raise AssertionError(
        "npm install did not generate package-lock.json."
    )
shutil.copy2(
    local_package_lock,
    FRONTEND_ROOT / "package-lock.json",
)

# Copy the successful local dist build back into the Drive project.
drive_dist_root = FRONTEND_ROOT / "dist"
if drive_dist_root.exists():
    shutil.rmtree(drive_dist_root)
shutil.copytree(local_dist_root, drive_dist_root)

# Copy the same deployable build into the submission directory.
if SUBMISSION_BUILD_ROOT.exists():
    shutil.rmtree(SUBMISSION_BUILD_ROOT)
shutil.copytree(local_dist_root, SUBMISSION_BUILD_ROOT)

for path in (
    drive_dist_root / "index.html",
    drive_dist_root / "data/app_data.json",
    SUBMISSION_BUILD_ROOT / "index.html",
    SUBMISSION_BUILD_ROOT / "data/app_data.json",
):
    if not path.exists() or path.stat().st_size == 0:
        raise AssertionError(
            f"Copied production build is missing: {path}"
        )

build_report = {
    "generated_utc": utc_now(),
    "node_version": node_version,
    "npm_version": npm_version,
    "build_strategy": (
        "npm install and Vite build executed on Colab local "
        "filesystem; package-lock and dist copied to Drive"
    ),
    "local_build_root": str(LOCAL_BUILD_ROOT),
    "npm_install_seconds": round(
        install_seconds,
        3,
    ),
    "frontend_build_seconds": round(
        build_seconds,
        3,
    ),
    "build_passed": True,
    "dist_file_count": sum(
        1
        for path in local_dist_root.rglob("*")
        if path.is_file()
    ),
    "drive_dist_root": drive_dist_root.relative_to(
        PROJECT_ROOT
    ).as_posix(),
    "submission_build_root": (
        SUBMISSION_BUILD_ROOT.relative_to(
            PROJECT_ROOT
        ).as_posix()
    ),
}
write_json(BUILD_REPORT_PATH, build_report)

print("=" * 104)
print("✅ Colab local-filesystem frontend build passed")
print(f"✅ Node.js {node_version}; npm {npm_version}")
print(f"✅ npm install: {install_seconds:.1f} s")
print(f"✅ Production build: {build_seconds:.1f} s")
print(
    "✅ package-lock.json copied to: "
    f"{FRONTEND_ROOT / 'package-lock.json'}"
)
print(f"✅ Drive dist build: {drive_dist_root}")
print(f"✅ Deployable build: {SUBMISSION_BUILD_ROOT}")
print(
    "ℹ️ node_modules remains local to /content and is not "
    "copied to Google Drive."
)
print("=" * 104)

✅ Colab local-filesystem frontend build passed
✅ Node.js v20.19.0; npm 10.8.2
✅ npm install: 7.9 s
✅ Production build: 4.5 s
✅ package-lock.json copied to: /content/drive/MyDrive/neurofhir-qc/app/frontend/package-lock.json
✅ Drive dist build: /content/drive/MyDrive/neurofhir-qc/app/frontend/dist
✅ Deployable build: /content/drive/MyDrive/neurofhir-qc/submission/reviewer_application
ℹ️ node_modules remains local to /content and is not copied to Google Drive.


In [6]:
# Cell 6 — Package source, audit the app, and update the manifest

required_files = [
    APP_DATA_PATH,
    FRONTEND_ROOT / "package.json",
    FRONTEND_ROOT / "package-lock.json",
    FRONTEND_ROOT / "vite.config.ts",
    FRONTEND_ROOT / "src/App.tsx",
    FRONTEND_ROOT / "src/styles.css",
    BACKEND_ROOT / "app/main.py",
    BACKEND_ROOT / "requirements.txt",
    APP_ROOT / "docker-compose.yml",
    APP_ROOT / "README.md",
    DEPLOYMENT_GUIDE_PATH,
    PROJECT_ROOT / ".github/workflows/deploy-reviewer-ui.yml",
    SCREEN_MATRIX_PATH,
    BUILD_REPORT_PATH,
    API_REPORT_PATH,
    SUBMISSION_BUILD_ROOT / "index.html",
]
missing = [str(path) for path in required_files if not path.exists() or path.stat().st_size == 0]
if missing:
    raise AssertionError("Notebook 11 artifacts missing:\n" + "\n".join(f" - {path}" for path in missing))

inventory_paths = sorted({
    path
    for root in (APP_ROOT, PROJECT_ROOT / ".github/workflows", SUBMISSION_BUILD_ROOT)
    for path in root.rglob("*")
    if path.is_file() and "node_modules" not in path.parts and path != ZIP_PATH
})
inventory = [{
    "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
    "size_bytes": path.stat().st_size,
    "sha256": sha256_file(path),
} for path in inventory_paths]
write_json(FILE_INVENTORY_PATH, {"generated_utc": utc_now(), "file_count": len(inventory), "files": inventory})

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
    for path in inventory_paths:
        archive.write(path, arcname=path.relative_to(PROJECT_ROOT).as_posix())
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise AssertionError(f"Application ZIP failed integrity: {bad_member}")
    zip_member_count = len(archive.namelist())

screen_matrix = load_json(SCREEN_MATRIX_PATH)
build_report = load_json(BUILD_REPORT_PATH)
api_report = load_json(API_REPORT_PATH)
final_gate = {
    "notebook_10_completed": nb10_audit.get("status") == "completed",
    "application_data_generated": APP_DATA_PATH.exists(),
    "six_screens_implemented": screen_matrix.get("implementation_rate") == 1.0,
    "frontend_build_passed": build_report.get("build_passed") is True,
    "backend_syntax_passed": api_report.get("backend_syntax_passed") is True,
    "static_build_created": (SUBMISSION_BUILD_ROOT / "index.html").exists(),
    "github_pages_workflow_created": (PROJECT_ROOT / ".github/workflows/deploy-reviewer-ui.yml").exists(),
    "source_archive_created": ZIP_PATH.exists(),
    "synthetic_data_boundary_preserved": True,
    "unsupported_smart_claim_blocked": True,
    "unsupported_clinical_claim_blocked": True,
}
failed = [key for key, passed in final_gate.items() if not passed]
if failed:
    raise AssertionError("Notebook 11 final gate failed: " + ", ".join(failed))

notebook_saved = NOTEBOOK_SAVE_PATH.exists() and NOTEBOOK_SAVE_PATH.stat().st_size > 0
final_audit = {
    "project_name": "NeuroFHIR-QC",
    "project_version": project_config.get("version", "0.1.0"),
    "notebook_number": "11",
    "notebook_filename": NOTEBOOK_FILENAME,
    "status": "completed",
    "audited_utc": utc_now(),
    "notebook_saved_in_drive": notebook_saved,
    "metrics": {
        "reviewer_screen_count": 6,
        "screen_implementation_rate": 1.0,
        "synthetic_case_count": len(app_data["cases"]),
        "representative_fhir_resource_count": len(app_data["representative_fhir_resources"]),
        "frontend_build_pass_rate": 1.0,
        "backend_syntax_pass_rate": 1.0,
        "deployable_static_build_created": True,
        "source_archive_member_count": zip_member_count,
    },
    "scope": {
        "react_reviewer_ui_generated": True,
        "fastapi_evidence_service_generated": True,
        "github_pages_workflow_generated": True,
        "docker_compose_generated": True,
        "production_smart_oauth_implemented": False,
        "real_clinician_usability_performed": False,
        "clinical_deployment_performed": False,
    },
    "safety": {
        "public_deidentified_imaging_only": True,
        "synthetic_fhir_only": True,
        "real_patient_data_used": False,
        "live_review_writeback_endpoint_exposed": False,
        "unsupported_clinical_claims_blocked": True,
    },
    "final_gate": final_gate,
    "output_paths": {
        "application_source_root": APP_ROOT.relative_to(PROJECT_ROOT).as_posix(),
        "static_build": SUBMISSION_BUILD_ROOT.relative_to(PROJECT_ROOT).as_posix(),
        "source_zip": ZIP_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "deployment_guide": DEPLOYMENT_GUIDE_PATH.relative_to(PROJECT_ROOT).as_posix(),
    },
    "next_step": "Commit and deploy the reviewer UI, capture screenshots, record the demo video, and run honest usability testing if feasible.",
}
write_json(AUDIT_JSON_PATH, final_audit)
AUDIT_MD_PATH.write_text(
    "# Notebook 11 — Reviewer Application UI\n\n"
    "**Status:** completed  \n"
    f"**Audited:** {final_audit['audited_utc']}\n\n"
    "## Implemented\n\n"
    "- React + TypeScript reviewer app\n"
    "- Six competition screens\n"
    "- FastAPI read-only evidence service\n"
    "- Docker Compose\n"
    "- GitHub Pages workflow\n"
    "- Production build and source ZIP\n\n"
    "Production SMART OAuth, clinical deployment, real clinician usability, and clinical validation are not claimed.\n",
    encoding="utf-8",
)

def entries(manifest: Any) -> list[dict[str, Any]]:
    if isinstance(manifest, list):
        return manifest
    if isinstance(manifest, dict):
        for key in ("notebooks", "entries", "workflow"):
            value = manifest.get(key)
            if isinstance(value, list):
                return value
        manifest["notebooks"] = []
        return manifest["notebooks"]
    raise ValueError("Unrecognized notebook manifest structure.")

rows = entries(notebook_manifest)
entry = next((row for row in rows if str(row.get("notebook_number") or row.get("number") or "").zfill(2) == "11"), None)
if entry is None:
    entry = {"notebook_number": "11", "filename": NOTEBOOK_FILENAME, "title": "Reviewer Application UI"}
    rows.append(entry)
entry.update({
    "status": "completed",
    "completed_utc": final_audit["audited_utc"],
    "reviewer_screen_count": 6,
    "frontend_build_pass_rate": 1.0,
    "backend_syntax_pass_rate": 1.0,
    "static_build": SUBMISSION_BUILD_ROOT.relative_to(PROJECT_ROOT).as_posix(),
    "source_zip": ZIP_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "audit_path": AUDIT_JSON_PATH.relative_to(PROJECT_ROOT).as_posix(),
})
write_json(NOTEBOOK_MANIFEST_PATH, notebook_manifest)

print("=" * 104)
print("✅ Notebook 11 Reviewer Application UI completed")
print("✅ Six reviewer screens implemented")
print("✅ React production build passed")
print("✅ FastAPI source validation passed")
print(f"📦 Static build: {SUBMISSION_BUILD_ROOT}")
print(f"📦 Source ZIP: {ZIP_PATH}")
print(f"✅ Audit JSON: {AUDIT_JSON_PATH}")
if not notebook_saved:
    print(f"⚠️ Save the executed notebook to {NOTEBOOK_SAVE_PATH}")
print("➡️ Next: deploy, capture screenshots, record video, and test usability")
print("=" * 104)

✅ Notebook 11 Reviewer Application UI completed
✅ Six reviewer screens implemented
✅ React production build passed
✅ FastAPI source validation passed
📦 Static build: /content/drive/MyDrive/neurofhir-qc/submission/reviewer_application
📦 Source ZIP: /content/drive/MyDrive/neurofhir-qc/submission/NeuroFHIR_QC_Reviewer_Application_Source.zip
✅ Audit JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_11_reviewer_application_ui_audit.json
⚠️ Save the executed notebook to /content/drive/MyDrive/neurofhir-qc/notebooks/11_NeuroFHIR_QC_Reviewer_Application_UI.ipynb
➡️ Next: deploy, capture screenshots, record video, and test usability
